# 多头注意力
:label:`sec_multihead-attention`

在实践中，当给定相同的查询、键和值的集合时，
我们希望模型可以基于相同的注意力机制学习到不同的行为，
然后将不同的行为作为知识组合起来，
捕获序列内各种范围的依赖关系
（例如，短距离依赖和长距离依赖关系）。
因此，允许注意力机制组合使用查询、键和值的不同
*子空间表示*（representation subspaces）可能是有益的。

为此，与其只使用单独一个注意力汇聚，
我们可以用独立学习得到的$h$组不同的
*线性投影*（linear projections）来变换查询、键和值。
然后，这$h$组变换后的查询、键和值将并行地送到注意力汇聚中。
最后，**将这$h$个注意力汇聚的输出拼接在一起**，
并且通过另一个可以学习的线性投影进行变换，
以产生最终输出。
这种设计被称为*多头注意力*（multihead attention）
 :cite:`Vaswani.Shazeer.Parmar.ea.2017`。
对于$h$个注意力汇聚输出，每一个注意力汇聚都被称作一个*头*（head）。
 :numref:`fig_multi-head-attention`
展示了使用全连接层来实现可学习的线性变换的多头注意力。

![多头注意力：多个头连结然后线性变换](../img/multi-head-attention.svg)
:label:`fig_multi-head-attention`



## 模型

在实现多头注意力之前，让我们用数学语言将这个模型形式化地描述出来。
给定查询$\mathbf{q} \in \mathbb{R}^{d_q}$、
键$\mathbf{k} \in \mathbb{R}^{d_k}$和
值$\mathbf{v} \in \mathbb{R}^{d_v}$，
每个注意力头$\mathbf{h}_i$（$i = 1, \ldots, h$）的计算方法为：

$$\mathbf{h}_i = f(\mathbf W_i^{(q)}\mathbf q, \mathbf W_i^{(k)}\mathbf k,\mathbf W_i^{(v)}\mathbf v) \in \mathbb R^{p_v},$$

其中，可学习的参数包括
$\mathbf W_i^{(q)}\in\mathbb R^{p_q\times d_q}$、
$\mathbf W_i^{(k)}\in\mathbb R^{p_k\times d_k}$和
$\mathbf W_i^{(v)}\in\mathbb R^{p_v\times d_v}$，
以及代表注意力汇聚的函数$f$。
$f$可以是 :numref:`sec_attention-scoring-functions`中的
加性注意力和缩放点积注意力。
多头注意力的输出需要经过另一个线性转换，
它对应着$h$个头连结后的结果，因此其可学习参数是
$\mathbf W_o\in\mathbb R^{p_o\times h p_v}$：

$$\mathbf W_o \begin{bmatrix}\mathbf h_1\\\vdots\\\mathbf h_h\end{bmatrix} \in \mathbb{R}^{p_o}.$$

*基于这种设计，每个头都可能会关注输入的不同部分，
可以表示比简单加权平均值更复杂的函数*。


对于transformer架构来说，多头注意力机制和CNN中的通道数是否有类似的地方，即能否将多头注意力的每一头对应于CNN中的每一个通道？
如果这样的话tansformer将多个注意力concat的操作能否类似的看成成1*1卷积
这种concat操作对于多通道数的CNN是否也有效？如果有效从计算性能和结果优化的角度来看 1*1卷积和concat全连接 哪个有优势？

---
**Transformer 的多头注意力**（Multi-Head Attention, MHA） 与 **CNN 的通道机制** 进行类比，并进一步思考 `concat` 与 `1×1 卷积` 的关系——这触及了深度学习中**表示解耦、特征融合与计算效率**的核心思想。

我们来逐层分析：



### ✅ 一、多头注意力 vs CNN 通道：**类比是否成立**？

#### 🔹 相似点（类比合理之处）：
| 维度 | Multi-Head Attention | CNN |
|------|------------------------|-----|
| **并行子空间** | 每个“头”在独立的线性投影子空间中计算注意力 | 每个“通道”提取不同类型的局部特征（边缘、纹理等） |
| **特征多样性** | 不同头可关注不同位置/语义关系（如语法 vs 语义） | 不同通道响应不同模式 |
| **融合方式** | 多头输出拼接（concat）后线性变换 | 1×1 卷积融合通道信息 |

> ✅ **从“多视角特征提取 + 融合”的角度看，类比是合理的**。

#### 🔹 关键区别（不能完全等同）：
| 方面 | MHA | CNN |
|------|-----|-----|
| **感受野** | 全局（每个 token 可 attend 所有其他 token） | 局部（由卷积核大小决定） |
| **交互方式** | 基于内容的动态权重（query-key 相似度） | 固定权重（学习到的卷积核） |
| **位置感知** | 需显式位置编码 | 卷积天然保留**空间结构** |

> ⚠️ **MHA 的“头”是语义/关系导向的，CNN 通道是空间/模式导向的**。

---

### ✅ 二、多头拼接（concat） ≈ 1×1 卷积？**部分成立**！

#### 🔸 MHA 的标准流程：
```python
# 假设 d_model = 512, num_heads = 8 → 每头 dim = 64
multi_head_output = concat(head_1, head_2, ..., head_8)  # (..., 512)
output = Linear(multi_head_output)                       # (..., 512)
```

#### 🔸 CNN 中的等效操作：
```python
# 输入: (B, C_in=512, H, W)
x = conv1x1(x)  # 输出: (B, C_out=512, H, W)
```

#### 📌 关键观察：
- **`concat + Linear` 在数学上等价于一个跨头的线性变换**
- **1×1 卷积也是跨通道的线性变换**（无空间混合）

> ✅ **从“跨子空间线性融合”的角度看，两者功能相似**！

事实上，**你可以用 1×1 卷积实现 MHA 的最后一步**（如果输入是 2D 图像格式）。

---

### ✅ 三、反过来：CNN 中能否用 `concat + FC` 替代 1×1 卷积？

#### 场景：假设你有多个并行分支（如 Inception 模块），每支输出 `C_i` 通道，总输出需融合为 `C_out`。

- 方案 A：1×1 卷积（标准做法）
```python
x = torch.cat([branch1, branch2, ...], dim=1)  # (B, C_total, H, W)
x = conv1x1(x)  # (B, C_out, H, W)
```

- 方案 B：全局展平 + 全连接（FC）
```python
x = torch.cat([branch1, branch2, ...], dim=1)
x = x.view(B, -1)          # (B, C_total * H * W)
x = fc(x)                  # (B, C_out * H * W)
x = x.view(B, C_out, H, W)
```

❌ 问题：**FC 会破坏空间结构**！
- FC 将 `(H, W)` 视为独立维度，无法保证平移等变性
- 参数量爆炸：`C_total * H * W * C_out * H * W`

> 🚫 **在 CNN 中，`concat + FC` 通常不可行**（除非 H=W=1，如 ViT 的 token）

---

### ✅ 四、性能与优化对比：1×1 Conv vs Concat+Linear

| 维度 | 1×1 卷积 | Concat + Linear |
|------|---------|------------------|
| **参数量** | $C_{\text{in}} \times C_{\text{out}}$ | 同左（若输入展平后维度相同） |
| **计算量** | $O(C_{\text{in}} C_{\text{out}} HW)$ | 同左（但常数更大） |
| **空间结构保持** | ✅ 是 | ❌ 否（除非输入已是 1D） |
| **硬件优化** | ✅ cuDNN 高度优化 | ⚠️ 通用 GEMM，略慢 |
| **适用场景** | CNN（2D/3D 特征图） | Transformer（1D token 序列） |

#### 📌 结论：
- **在 1D 序列**（如 Transformer）  
  → `concat + Linear` 是自然且高效的选择
- **在 2D/3D 空间数据**（如图像）  
  → **1×1 卷积是唯一合理选择**（保持空间局部性）

---

### ✅ 五、更深层理解：为什么 Transformer 用 concat + Linear？

因为 Transformer 的输入是 **1D token 序列**（无空间结构）：
- 每个 token 是一个向量（如 512D）
- 多头输出：每个头给每个 token 一个子向量（如 64D）
- 拼接后仍是每个 token 一个 512D 向量
- **Linear 层作用于每个 token 独立**（相当于 1×1 卷积在 1D 上的特例）

> 💡 **可以说：Transformer 中的 `Linear` 就是 1D 版的 1×1 卷积**！

---

### ✅ 总结

| 问题 | 回答 |
|------|------|
| **多头 ≈ CNN 通道**？ | ✅ 功能类比成立（多视角特征提取），但机制不同（全局动态 vs 局部静态） |
| **Concat ≈ 1×1 Conv**？ | ✅ 在 1D 序列上，`concat + Linear` ≈ 1×1 卷积；在 2D 上不等价 |
| **CNN 能否用 concat+FC**？ | ❌ 会破坏空间结构，参数爆炸，不实用 |
| **哪个更有优势**？ | **取决于数据结构**：• 1D 序列 → `Linear`• 2D/3D → `1×1 Conv` |

> 🌟 **核心原则**：  
> **特征融合操作必须尊重输入数据的拓扑结构**。  
> Transformer 处理的是“无序但带位置编码的集合”，CNN 处理的是“有序网格”，因此融合策略自然不同。


In [1]:
import math
import torch
from torch import nn
from myd2l import torch as d2l

## 实现

在实现过程中通常[**选择缩放点积注意力作为每一个注意力头**]。
为了避免计算代价和参数代价的大幅增长，
我们设定$p_q = p_k = p_v = p_o / h$，隐藏层的个数。
值得注意的是，如果将查询、键和值的线性变换的输出数量设置为
$p_q h = p_k h = p_v h = p_o$，
则可以并行计算$h$个头。
在下面的实现中，$p_o$是通过参数`num_hiddens`指定的。


In [2]:
#@save
class MultiHeadAttention(nn.Module):
    """多头注意力"""
    def __init__(self, key_size, query_size, value_size, num_hiddens,
                 num_heads, dropout, bias=False, **kwargs):
        super(MultiHeadAttention, self).__init__(**kwargs)
        self.num_heads = num_heads
        self.attention = d2l.DotProductAttention(dropout)
        self.W_q = nn.Linear(query_size, num_hiddens, bias=bias)
        self.W_k = nn.Linear(key_size, num_hiddens, bias=bias)
        self.W_v = nn.Linear(value_size, num_hiddens, bias=bias)
        self.W_o = nn.Linear(num_hiddens, num_hiddens, bias=bias)

    def forward(self, queries, keys, values, valid_lens):
        # queries，keys，values的形状:
        # (batch_size，num_q，num_hiddens)
        # valid_lens　的形状:
        # (batch_size，)或(batch_size，num_q)
        # 经过变换后，输出的queries，keys，values　的形状:
        # (batch_size*num_heads，num_q，num_hiddens/num_heads)
        queries = transpose_qkv(self.W_q(queries), self.num_heads)
        keys = transpose_qkv(self.W_k(keys), self.num_heads)
        values = transpose_qkv(self.W_v(values), self.num_heads)

        if valid_lens is not None:
            # 在轴0，将第一项（标量或者矢量）复制num_heads次，
            # 然后如此复制第二项，然后诸如此类。
            valid_lens = torch.repeat_interleave(
                valid_lens, repeats=self.num_heads, dim=0)

        # output的形状:(batch_size*num_heads，查询的个数，
        # num_hiddens/num_heads)
        output = self.attention(queries, keys, values, valid_lens)

        # output_concat的形状:(batch_size，查询的个数，num_hiddens)
        output_concat = transpose_output(output, self.num_heads)
        return self.W_o(output_concat)

为了能够[**使多个头并行计算**]，
上面的`MultiHeadAttention`类将使用下面定义的两个转置函数。
具体来说，`transpose_output`函数反转了`transpose_qkv`函数的操作。


In [3]:
#@save
def transpose_qkv(X, num_heads):
    """为了多注意力头的并行计算而变换形状"""
    # 输入X的形状:(batch_size，查询或者“键－值”对的个数，num_hiddens)
    # 输出X的形状:(batch_size，查询或者“键－值”对的个数，num_heads，
    # num_hiddens/num_heads)
    X = X.reshape(X.shape[0], X.shape[1], num_heads, -1)

    # 输出X的形状:(batch_size，num_heads，查询或者“键－值”对的个数,
    # num_hiddens/num_heads)
    X = X.permute(0, 2, 1, 3)

    # 最终输出的形状:(batch_size*num_heads,查询或者“键－值”对的个数,
    # num_hiddens/num_heads)
    return X.reshape(-1, X.shape[2], X.shape[3])


#@save
def transpose_output(X, num_heads):
    """逆转transpose_qkv函数的操作"""
    X = X.reshape(-1, num_heads, X.shape[1], X.shape[2])
    X = X.permute(0, 2, 1, 3)
    return X.reshape(X.shape[0], X.shape[1], -1)

下面使用键和值相同的小例子来[**测试**]我们编写的`MultiHeadAttention`类。
多头注意力输出的形状是（`batch_size`，`num_queries`，`num_hiddens`）。


In [4]:
num_hiddens, num_heads = 100, 5
attention = MultiHeadAttention(num_hiddens, num_hiddens, num_hiddens,
                               num_hiddens, num_heads, 0.5)
attention.eval()

MultiHeadAttention(
  (attention): DotProductAttention(
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (W_q): Linear(in_features=100, out_features=100, bias=False)
  (W_k): Linear(in_features=100, out_features=100, bias=False)
  (W_v): Linear(in_features=100, out_features=100, bias=False)
  (W_o): Linear(in_features=100, out_features=100, bias=False)
)

In [5]:
batch_size, num_queries = 2, 4
num_kvpairs, valid_lens =  6, torch.tensor([3, 2])
X = torch.ones((batch_size, num_queries, num_hiddens))
Y = torch.ones((batch_size, num_kvpairs, num_hiddens))
attention(X, Y, Y, valid_lens).shape

torch.Size([2, 4, 100])

## 小结

* 多头注意力融合了来自于多个注意力汇聚的不同知识，这些知识的不同来源于相同的查询、键和值的不同的子空间表示。
* 基于适当的张量操作，可以实现多头注意力的并行计算。

## 练习

1. 分别可视化这个实验中的多个头的注意力权重。
1. 假设有一个完成训练的基于多头注意力的模型，现在希望修剪最不重要的注意力头以提高预测速度。如何设计实验来衡量注意力头的重要性呢？


[Discussions](https://discuss.d2l.ai/t/5758)
